# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tanay1122/flyrank-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: ranking / scoring** — the framing-skill table's *"Which ones first?"* row. Lane 2's question ("which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?") is answered by an **ordered queue**, not by a pile of yes/no output. So the model's output is a per-page **priority score** plus reason codes; underneath it, a classification helper (each page's decline probability) feeds the score, and the suggested action per page (refresh / expand / protect / prune / monitor) is a **policy layer I write** — a rule on top of evidence, not a prediction.

Not clustering: clustering produces named groups — a lens, not an ordering; no true labels exist to justify making a cluster label a decision product. Not "EDA-only": exploratory summaries matter and come first in the weekly plan (that is notebook w04), but the decision consumer needs a full ordering of ~20,883 meaningful candidates every week — no summary produces that. The repo itself agrees about the task type: its model card picks winners by **ranking** quality — the committed report literally says "Best model: `random_forest` selected by `precision_at_50`" (printed below).

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Load the lane's slice (starter starter 30k pages), confirm the arena, and show the repo's own task-type vote.
from pathlib import Path
import subprocess

import pandas as pd

REPO_URL = "https://github.com/tanay1122/flyrank-ml.git"
DATA_PATH = Path("data") / "raw" / "content_refresh_anonymized.csv"


def find_repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / DATA_PATH).exists():
            return candidate
    # Cloud-runtime fallback: e.g. the Colab badge opens just the notebook, not the repo.
    for parent in [Path("/content"), Path.cwd()]:
        target = parent / "flyrank-ml"
        if not target.parent.exists():
            continue
        if not (target / DATA_PATH).exists():
            print(f"Starter CSV not found locally; cloning the public repo into {target} ...")
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)], check=False, capture_output=True)
        if (target / DATA_PATH).exists():
            return target
    raise FileNotFoundError("Starter CSV not found - open/run this notebook from inside the flyrank-ml repo.")


REPO = find_repo_root()
df = pd.read_csv(REPO / DATA_PATH)
print(f"Starter slice: {df.shape[0]:,} rows x {df.shape[1]} columns | {df['client_id'].nunique()} clients | "
      f"{df['content_id'].nunique():,} unique content_id (no dupes)")

import re
report = (REPO / "outputs" / "model_report.md").read_text(encoding="utf-8")
for line in report.splitlines():
    if line.startswith("Best model:"):
        print("Repo's own task-type vote ->", line.strip())

Starter slice: 30,000 rows x 44 columns | 32 clients | 30,000 unique content_id (no dupes)
Repo's own task-type vote -> Best model: `random_forest` selected by `precision_at_50`.


## 2. Target or proxy

**Week-2 target (the starter's proxy):** `is_declining_label = (trend_direction == "down")`. Its origin is honestly limited: it is a **defined rule, not an observed outcome** — last-30d vs previous-30d impressions with a -20% / +20% cut, computed inside the snapshot's own window. It is a stand-in for "this page looks like it is sliding," and the model that learns it is partly learning the window's movement patterns, not the future. Because the label is derived from those columns, `trend_direction` and `trend_pct` can **never** be features (guard runs below, and it passed).

**The honest capstone target (observed):** a **future-window outcome** built from warehouse page-day facts: *features from the page's prior 90 days -> sustained decline over the NEXT 30 days*, where the outcome (magnitude + persistence + minimum volume + consolidation/seasonality checks) is **measured in a later window** that features never touch, and only the thresholds are my written definitions in the data contract (ML-04). The ranked queue remains the product; the future-decline label is the evaluation target that keeps the queue honest, and the starter proxy is its weak stand-in.

Three layers, kept separate on purpose: (1) the **score** — the primary ranking output; (2) the **decline label** — evaluation target (proxy today, future-window once the warehouse tables are in); (3) the **review action** — a policy written from reason codes, not predicted.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Label mechanics: base rate, the defining rule, the never-feature guard, and the target sketch.
import sys

import numpy as np

trend = df["trend_direction"].astype(str).str.lower()
y = (trend == "down").astype(int)
print(f"Label rule (from the data dictionary): down = last-30d impressions fell by >= 20% vs the previous 30d")
print(f"Proxy positives: {int(y.sum()):,} pages  ({y.mean() * 100:.1f}% base rate)")

sys.path.insert(0, str(REPO / "scripts"))
from ml_utils import MODEL_CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES

label_sources = {"trend_direction", "trend_pct", "is_declining_label"}
model_features = set(MODEL_NUMERIC_FEATURES) | set(MODEL_CATEGORICAL_FEATURES)
overlap = sorted(label_sources & model_features)
assert not overlap, f"label sources leaked into features: {overlap}"
print(f"Guard passed: none of {sorted(label_sources)} appear among the {len(model_features)} starter model features")

sketch = pd.DataFrame({
    "content_id (pseudonym)": df["content_id"].head(5).values,
    "starter_proxy_decline": y.head(5).values,
    "future_decline_next30d": np.nan,
})
print("Target sketch (what the columns would look like; the NaN column gets filled from warehouse page-day facts):")
print(sketch.to_string(index=False))

Label rule (from the data dictionary): down = last-30d impressions fell by >= 20% vs the previous 30d
Proxy positives: 16,262 pages  (54.2% base rate)
Guard passed: none of ['is_declining_label', 'trend_direction', 'trend_pct'] appear among the 26 starter model features
Target sketch (what the columns would look like; the NaN column gets filled from warehouse page-day facts):
content_id (pseudonym)  starter_proxy_decline  future_decline_next30d
  content_304f48230142                      1                     NaN
  content_a1fb4e703a9e                      1                     NaN
  content_9aa793d4d895                      1                     NaN
  content_331d6c4de07b                      0                     NaN
  content_d99b7a2d90ca                      1                     NaN


## 3. Success metric

**Primary metric: precision@K.** Of the K pages my queue puts first, how many actually carry the (label-defined) opportunity or problem. **K comes from the acting person's capacity** — about 20 pages/week, so I evaluate at K=20 and report at K=50 like the committed starter run. It is the defensible metric because it matches how the queue is consumed (top-down, against a fixed slot); the lane guide's own metric table says top-K beats generic accuracy for ranked queues, and the repo reinforces it by selecting its best model with `precision_at_50`. Secondary, context only: **average precision** (the whole ordering matters), and the **base rate** (54.2%) — the score a no-skill selector would hit — sits there to stop anyone celebrating small wins.

**What 'good' means, precisely:** beat the strongest *transparent hand rule* at the same K under the same evaluation protocol, and survive a by-hand read of my top 20. Anchors today: committed client-holdout numbers on the proxy label — baseline rules P@50 = 0.24, random forest = 0.74. Computed live below on the full 30k slice (no holdout; directional context, not head-to-head): the starter's hand-weighted composite score gets P@50 = 0.34, and a single hand rule ("stale-first, only among visible pages") already reaches P@50 = 0.82 on this toy proxy. Careful words: full-slice and holdout numbers are not directly comparable; the real bar is measured on the future-window label later — and if a simple rule wins there, keeping the rule is the honest outcome, not a failure.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Compute the metric TODAY, with no training: mirror the starter's composite baseline exactly (scripts/02_baseline_score.py math),
# rank the whole slice, and read precision@K off it with stable, deterministic tie-breaks.
import numpy as np
import re

import sys

sys.path.insert(0, str(REPO / "scripts"))
from ml_utils import normalize, percentile_rank

def precision_at_k_stable(scores: pd.Series, truth: pd.Series, k: int) -> float:
    order = np.argsort(-scores.to_numpy(dtype=float), kind="stable")[:k]
    return float(truth.iloc[order].mean())

x = df.copy()
num_fill = [c for c in ["search_volume", "competition", "cpc", "word_count", "char_count", "impressions_90d",
                        "days_since_last_update", "ctr", "avg_position", "sessions_90d", "engagement_rate",
                        "scroll_rate"] if c in x.columns]
for col in num_fill:
    x[col] = pd.to_numeric(x[col], errors="coerce").fillna(0)

x["visibility_score"] = percentile_rank(np.log1p(x["impressions_90d"]))
x["freshness_risk_score"] = percentile_rank(x["days_since_last_update"])
x["position_opportunity_score"] = (
    (1 - normalize(x["avg_position"].clip(lower=1, upper=50)))
    * x["visibility_score"]
    * (x["avg_position"] > 0).astype(int)
)
x["depth_gap_score"] = (1 - percentile_rank(x["word_count"])) * x["visibility_score"]
x["baseline_refresh_score"] = (
    0.40 * x["visibility_score"] + 0.30 * x["freshness_risk_score"]
    + 0.25 * x["position_opportunity_score"] + 0.05 * x["depth_gap_score"]
).clip(0, 1)

print(f"Composite starter baseline (mirrored math, full 30k slice):")
print(f"  precision@20 = {precision_at_k_stable(x['baseline_refresh_score'], y, 20):.2f}"
      f"   precision@50 = {precision_at_k_stable(x['baseline_refresh_score'], y, 50):.2f}")
print(f"  no-skill reference (base rate) = {y.mean():.2f}")

report = (REPO / "outputs" / "model_report.md").read_text(encoding="utf-8")
print("Committed client-holdout comparison (best model was chosen by precision@50):")
for line in report.splitlines():
    if re.match(r"^\|\s*(baseline_rules|logistic_regression|decision_tree|random_forest)\b", line):
        print(" ", line.replace("|", " ").strip())

Composite starter baseline (mirrored math, full 30k slice):
  precision@20 = 0.35   precision@50 = 0.34
  no-skill reference (base rate) = 0.54
Committed client-holdout comparison (best model was chosen by precision@50):
  decision_tree   0.742   0.575   0.540   0.716   0.634
  logistic_regression   0.700   0.522   0.400   0.567   0.566
  random_forest   0.750   0.618   0.740   0.744   0.640
  baseline_rules   0.627   0.468   0.240   -   -


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item (a page).** The starter slice is the reviewer's decision surface: 30,000 pages across 32 clients, every column describing the same page over one trailing-90-day window, and every row already inside the pipeline's pool (at least 1 impression, at least 90 days old, no duplicate `content_id`) — the checks are printed below. The 5-page extract under the checks is exactly the shape the reviewer consumes: identifiers to pull the record, window totals, plus the state columns the ranking uses.

The warehouse upgrade later changes the grain deliberately: `fact_content_daily_performance` is **client x content x day** (page-day rows), from which per-page prior-window features and next-window outcomes are built — the same decision surface, just built from days instead of delivered pre-aggregated.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Prove the grain: one row = one page, and show a real extract.
import pandas as pd

pool = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id")
print(f"Pipeline-pool check: keep impressions_90d > 0 AND content_age_days >= 90, dedup by content_id -> "
      f"{len(pool):,} rows ({len(pool) == len(df)}, i.e. the whole slice is already the pool)")

show_cols = ["content_id", "client_id", "impressions_90d", "clicks_90d", "ctr", "avg_position",
             "sessions_90d", "engagement_rate", "content_age_days", "days_since_last_update", "trend_direction"]
extract = df[show_cols].head(5).copy()
extract["ctr"] = extract["ctr"].round(2)
extract["avg_position"] = extract["avg_position"].round(1)
print(f"One row = one page. Real extract (identifiers are pseudonyms, numbers are 90-day window totals):")
print(extract.to_string(index=False))

Pipeline-pool check: keep impressions_90d > 0 AND content_age_days >= 90, dedup by content_id -> 30,000 rows (True, i.e. the whole slice is already the pool)
One row = one page. Real extract (identifiers are pseudonyms, numbers are 90-day window totals):
          content_id         client_id  impressions_90d  clicks_90d  ctr  avg_position  sessions_90d  engagement_rate  content_age_days  days_since_last_update trend_direction
content_304f48230142 client_f369cb89fc             3803          29 0.76          10.6            17             5.88               187                      20            down
content_a1fb4e703a9e client_4e07408562            15320           7 0.05          20.3             9             0.00               445                      25            down
content_9aa793d4d895 client_7f2253d7e2            12581          11 0.09          36.5            11             0.00               141                      20            down
content_331d6c4de07b client_19581e27de   

## 5. Why ML beats a fixed rule here

The pattern the reviewer needs is **real but too messy to hand-write** — the framing-skill's own test for when ML earns its place:

1. **Many signals, heavy-tailed, interacting.** The 18 starter features move together in ways that shift with demand: the traffic is heavily concentrated — the ~16.7k pages (55.8%) with >= 500 impressions hold ~99% of logged impressions — and decline is not a monotone function of any one signal (the cell table below: "stale" barely matters among low-demand pages, 47.5% vs 42.0%, but among visible pages it is a 59.5% -> 94.1% jump).
2. **A hand rule has fixed reach.** The strongest single hand rule found ("stale-first, visible-only") shines in a niche that contains just **17 pages in total**. A 20-pages-per-week queue exhausts it in week one and then needs new winners across other archetypes (CTR-gap, page-one decay, thin-but-visible). Hand weights (0.40/0.30/0.25/0.05) frozen in code cannot re-tune themselves per window.
3. **Same data, same metric, more capacity -> better ranking.** The committed client-holdout ladder printed in section 3 (rules 0.24 -> logistic 0.40 -> tree 0.54 -> random forest 0.74 on precision@50) is exactly what "too messy for an if-statement" looks like when measured.
4. **The honest asterisk:** on the CURRENT proxy label a well-aimed hand rule already reaches P@50 = 0.82 on the full slice — protocols differ, so this is not a defeat of ML; it is more evidence the proxy label is toy-simple. The decisive test happens on the future-window label, where single-window hand rules cannot even express the outcome; if a rule still wins there, the queue keeps the rule — an honest result, not a failure.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Rules vs reality: heavy tail, single-rule precision, the interaction table, and the capacity ceiling.
import numpy as np
import pandas as pd

imp, days = df["impressions_90d"], df["days_since_last_update"]

big = imp >= 500
print(f"Heavy tail: {int(big.sum()):,} pages ({big.mean() * 100:.1f}%) hold {imp[big].sum() / imp.sum() * 100:.1f}% of logged impressions")

def precision_at_k_stable(scores: pd.Series, truth: pd.Series, k: int) -> float:
    order = np.argsort(-scores.to_numpy(dtype=float), kind="stable")[:k]
    return float(truth.iloc[order].mean())

rules = {
    "biggest first (sort by impressions)": imp,
    "stale first (sort by days_since_last_update, no demand floor)": days,
    "stale first, visible only (imp >= 500)": days.where(big, -1.0),
}
for name, scores in rules.items():
    print(f"  {name:58s} P@50 = {precision_at_k_stable(scores, y, 50):.2f}")

pairs = pd.DataFrame({
    "demand": np.where(imp >= 500, "imp>=500", "imp<500"),
    "staleness": np.where(days >= 180, "stale180+", "fresher"),
}).assign(down=y)
print("Interaction table - decline rate (pages per cell):")
for demand in ["imp<500", "imp>=500"]:
    line = f"  {demand:>8}:"
    for stale_col in ["fresher", "stale180+"]:
        mask = (pairs["demand"] == demand) & (pairs["staleness"] == stale_col)
        line += f"  {stale_col} = {y[mask].mean():.3f} ({int(mask.sum()):,})"
    print(line)

niche = big & (days >= 180)
print(f"Capacity ceiling of the strongest hand rule: its best cut (imp>=500 & stale180+) holds just "
      f"{int(niche.sum())} pages in the whole slice — a 20-per-week queue eats it in week one.")

Heavy tail: 16,726 pages (55.8%) hold 99.0% of logged impressions
  biggest first (sort by impressions)                        P@50 = 0.42
  stale first (sort by days_since_last_update, no demand floor) P@50 = 0.52
  stale first, visible only (imp >= 500)                     P@50 = 0.82
Interaction table - decline rate (pages per cell):
   imp<500:  fresher = 0.475 (13,117)  stale180+ = 0.420 (157)
  imp>=500:  fresher = 0.595 (16,709)  stale180+ = 0.941 (17)
Capacity ceiling of the strongest hand rule: its best cut (imp>=500 & stale180+) holds just 17 pages in the whole slice — a 20-per-week queue eats it in week one.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.